# 01 Build Sales Fact Table (Silver Layer)

This notebook reads raw transaction records from the Bronze layer (`train.parquet`), applies standard column names, casts types correctly, and drops raw primary keys that are not needed in downstream feature engineering.

In [1]:
import os
import time
from pathlib import Path
import gc
import psutil
import shutil
import pyarrow as pa
import pyarrow.parquet as pq

print("Libraries successfully imported!")

Libraries successfully imported!


In [2]:
# Locate lakehouse root dynamically
lakehouse_root = None
for p in [Path.cwd()] + list(Path.cwd().parents):
    if (p / "data_lake").is_dir() and (p / "notebooks").is_dir():
        lakehouse_root = p
        break

if lakehouse_root is None:
    for search_path in [
        "/content/Demand_forecasting/inventory-lakehouse",
        "/content/inventory-lakehouse",
        "/content/drive/MyDrive/Demand_forecasting/inventory-lakehouse"
    ]:
        if Path(search_path).is_dir():
            lakehouse_root = Path(search_path)
            break

if lakehouse_root is None:
    lakehouse_root = Path("D:/Demand_forecasting/inventory-lakehouse")

project_root = lakehouse_root.parent
BRONZE_DIR = lakehouse_root / "data_lake" / "bronze"
SILVER_DIR = lakehouse_root / "data_lake" / "silver"

SILVER_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project Root: {project_root}")
print(f"Bronze Dir:   {BRONZE_DIR}")
print(f"Silver Dir:   {SILVER_DIR}")

Project Root: D:\Demand_forecasting
Bronze Dir:   D:\Demand_forecasting\inventory-lakehouse\data_lake\bronze
Silver Dir:   D:\Demand_forecasting\inventory-lakehouse\data_lake\silver


In [3]:
start_time = time.time()
sales_fact_out_dir = SILVER_DIR / "sales_fact"
if sales_fact_out_dir.exists():
    shutil.rmtree(sales_fact_out_dir)
sales_fact_out_dir.mkdir(parents=True, exist_ok=True)

bronze_train_dir = BRONZE_DIR / "train.parquet"
train_parts = sorted(list(bronze_train_dir.glob("*.parquet")))

print("Starting transformation of train fact dataset...")
print(f"Found {len(train_parts)} Bronze partition parts to process.")

process = psutil.Process(os.getpid())
total_sales_rows = 0

for idx, part in enumerate(train_parts):
    part_start = time.time()
    table = pq.read_table(part)
    
    # Rename columns to standard silver schema
    new_names = ["id", "sales_date", "store_id", "sku_id", "units_sold", "is_promotion"]
    table = table.rename_columns(new_names)
    table = table.drop(["id"])
    
    # Write part
    part_out_path = sales_fact_out_dir / f"part_{idx}.parquet"
    pq.write_table(table, part_out_path, compression='snappy')
    
    part_rows = table.num_rows
    total_sales_rows += part_rows
    
    del table
    gc.collect()
    pa.default_memory_pool().release_unused()
    
    part_duration = time.time() - part_start
    print(f"  - Ingested part_{idx}.parquet: {part_rows:,} rows in {part_duration:.2f}s. RAM: {psutil.Process().memory_info().rss // (1024*1024)} MB")

p1_duration = time.time() - start_time
print(f"\nsales_fact ingestion completed successfully!")
print(f"Total Rows:     {total_sales_rows:,}")
print(f"Total Duration: {p1_duration:.2f} seconds")

Starting transformation of train fact dataset...
Found 7 Bronze partition parts to process.
  - Ingested part_0.parquet: 20,008,861 rows in 1.76s. RAM: 941.32 MB
  - Ingested part_1.parquet: 20,059,730 rows in 2.08s. RAM: 1.37 GB
  - Ingested part_2.parquet: 19,948,499 rows in 2.42s. RAM: 1.70 GB
  - Ingested part_3.parquet: 20,011,902 rows in 2.07s. RAM: 1.96 GB
  - Ingested part_4.parquet: 20,035,041 rows in 2.06s. RAM: 2.07 GB
  - Ingested part_5.parquet: 19,944,836 rows in 2.15s. RAM: 2.32 GB
  - Ingested part_6.parquet: 5,488,171 rows in 0.62s. RAM: 2.40 GB

sales_fact ingestion completed successfully!
Total Rows:     125,497,040
Silver Size:    350.26 MB
Total Duration: 13.16 seconds
